# 03 — Full Q&A Pipeline: Nutrition Guide Bot

The **complete RAG pipeline** using LangChain's LCEL (LangChain Expression Language).

### What we build
```
User question
     │
     ▼
[FAISS retriever] ── top-3 chunks ──▶ [PromptTemplate]
                                             │
                                        [Groq LLM]
                                             │
                                        Answer (with sources)
```

### New concepts
| Concept | Description |
|---------|-------------|
| `HuggingFaceEmbeddings` | LangChain wrapper — embeds docs & queries in one API |
| `FAISS.from_documents` | Builds vector store directly from Document list |
| LCEL `|` pipe | Chain components: retriever → prompt → LLM → parser |
| `PromptTemplate` | Structured prompt that injects context + question |

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install Dependencies

In [ ]:
# !pip install langchain langchain-community langchain-groq langchain-huggingface faiss-cpu python-dotenv

## 2. Imports & API Key

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 3. Load and Chunk the Nutrition Guide

In [ ]:
text = Path("../data/nutrition_guide.txt").read_text(encoding="utf-8")
doc  = Document(page_content=text, metadata={"source": "nutrition_guide.txt"})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents([doc])
print(f"Created {len(chunks)} chunks from nutrition guide")

## 4. Build the Vector Store

`HuggingFaceEmbeddings` is the LangChain wrapper around `sentence-transformers`.
Using it lets FAISS handle both storage and query embedding in a unified API.

In [ ]:
print("Loading embeddings model (downloads ~90 MB first time)...")
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Building FAISS vector store...")
vectorstore = FAISS.from_documents(chunks, embedder)
print(f"Vector store ready — {vectorstore.index.ntotal} vectors")

# Save so we don't re-embed on every run
vectorstore.save_local("faiss_nutrition")
print("Saved to faiss_nutrition/")

## 5. Create the Retriever

A **retriever** wraps the vector store and handles the search.
`search_kwargs={"k": 3}` means "return the 3 most relevant chunks".

In [ ]:
# Load from disk (skip if already in memory)
vectorstore = FAISS.load_local(
    "faiss_nutrition", embedder,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready")

# Quick test
test_results = retriever.invoke("What are macronutrients?")
print(f"Test query returned {len(test_results)} chunks")
print(f"Top result preview: {test_results[0].page_content[:100]}...")

## 6. The Prompt Template

The prompt tells the LLM:
1. Use only the provided context
2. Answer the question clearly
3. Mention if the context doesn't contain enough information

In [ ]:
RAG_PROMPT = PromptTemplate.from_template(
    """You are a helpful nutrition assistant. Answer the question using ONLY the context below.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    """Join retrieved chunks into a single context string."""    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in docs
    )

print("Prompt template ready")

## 7. Build the RAG Chain (LCEL)

The **pipe `|` operator** chains components left-to-right.

```python
{"context": retriever | format_docs, "question": RunnablePassthrough()}
  |  prompt
  |  llm
  |  StrOutputParser()
```

Each step passes its output as input to the next.

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0.1)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain built!")

## 8. Ask Questions!

In [ ]:
questions = [
    "What are the three macronutrients and what do they do?",
    "How much water should I drink every day?",
    "What is dietary fibre and why does it matter?",
    "What foods are considered superfoods?",
]

for question in questions:
    print(f"\n{'═'*58}")
    print(f"  Q: {question}")
    print(f"{'═'*58}")
    answer = rag_chain.invoke(question)
    print(f"  A: {answer}")

## 9. Retrieval with Source Attribution

In [ ]:
def ask_with_sources(question: str) -> dict:
    """Returns both the answer and the source chunks used."""    docs    = retriever.invoke(question)
    context = format_docs(docs)

    answer = (
        {"context": lambda _: context, "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke(question)

    sources = list({d.metadata.get("source", "unknown") for d in docs})
    return {"answer": answer, "sources": sources, "num_chunks": len(docs)}

result = ask_with_sources("How much protein do I need daily?")
print(f"Answer : {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Chunks used: {result['num_chunks']}")

## 10. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| `HuggingFaceEmbeddings` | LangChain wrapper — consistent API for embed + retrieve |
| `FAISS.from_documents` | One-step: embed + index all chunks |
| LCEL `|` pipe | Compose retriever → prompt → LLM → parser as a single chain |
| `PromptTemplate` | Structured prompt ensures LLM uses context correctly |
| Source attribution | Track which chunks contributed to the answer |

**Next notebook →** load multiple documents with metadata filtering.